[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Why Classes


## What you will be able to do

Recognize when a group of variables that always travel together should become a class, and
recognize the more common case where a function, a dictionary or a module is the better answer.


## The idea

### The problem

You are tracking three weather stations. Each one has a name, a list of readings, and the unit
those readings were recorded in, because one team's instrument reports Fahrenheit and the other
two report Celsius.

So you write three variables per station, and functions that take them:

`average(readings, unit)`, `in_celsius(readings, unit)`, `report(name, readings, unit)`.

This works. It is ordinary Python and there is nothing wrong with it.

Then the problems start. A fourth attribute arrives, whether the instrument has been calibrated,
and every function signature has to change, along with every call. A second station means six
variables in your program rather than three, and a fourth station means twelve. And the moment
there is more than one station, nothing stops you calling `report(north_name, north_readings,
south_unit)`. Python will not complain. It will report a Norwegian station in Fahrenheit and hand
you a number that is wrong by forty degrees.

That last one is the real problem. The name, the readings and the unit describe one thing, and the
language has no idea. They are three separate values that you, and only you, are keeping in step.

### What a class is

> A **class** is a template that binds data together with the functions that operate on it.
> Creating an **object** from the class, also called an **instance**, produces a value carrying its
> own copy of that data, and the functions become **methods** you call on the object.

### Why it works that way

The gain is not shorter code. Written out, the class version of this program is slightly longer
than the functions were.

The gain is that a whole category of mistake stops being expressible. Once the unit lives on the
station, `north.average()` can only reach `north`'s unit, because that is the only unit it can
see. The mismatched call is not caught at runtime and it is not caught by a check you wrote. There
is simply no way to write it.

The second gain is that the station becomes a value. You can put stations in a list, sort that
list by their coldest reading, pass one to a function, and return one from a function. Three loose
variables cannot be put in a list without inventing a convention for how they line up, and a
convention is a thing that can be got wrong.

The cost is indirection. `self.readings` is a longer way of saying `readings`, and somebody reading
`north.report()` has to go and find the class to see what it does. On a program with three
variables and one function, that cost is larger than the gain, which is why the second half of this
notebook is about not writing a class.

Python does not require classes. Most Python code is functions in modules, and that is not a
failing to be corrected. The question is never whether to use classes; it is whether this
particular thing is one.

### Where you will meet this

Everywhere, in code you did not write. `Path` is a class, and so is `ZipFile`. Every exception you
have caught is a class. In the **Pandas** guide a `DataFrame` is a class and nearly everything you
do with one is a method call. You will read far more classes than you write, so knowing the shape
matters even if you decide you rarely need your own.

### What this notebook covers

The program written both ways, the mistake that only the first version allows, and three shapes
that look like classes and are better as something else. The mechanics of writing one, `__init__`
and `self`, come next.

### A first look

The same two facts, held two ways. There is nothing to run yet: read it, and read the output
underneath it.

```python
from statistics import mean

name, readings, unit = "Tromso", [-4.1, -2.6, -3.8], "C"
print(f"{name}: {mean(readings):.1f} {unit}")


class Station:
    def __init__(self, name, readings, unit):
        self.name = name
        self.readings = readings
        self.unit = unit

    def average(self):
        return f"{self.name}: {mean(self.readings):.1f} {self.unit}"


print(Station("Tromso", [-4.1, -2.6, -3.8], "C").average())
```

```
Tromso: -3.5 C
Tromso: -3.5 C
```

Identical output, and for one station the top version is plainly better. Keep that in mind while
reading the rest: the case for the class is not visible until there is more than one station.


## Setup

One import, and the readings this notebook works with.

- `mean` averages a list of numbers, so the examples can spend their length on structure rather
  than on adding numbers up by hand

The three stations are written as loose variables on purpose. That is the version the notebook
starts from.

**Run this cell before the rest of the notebook.**


In [1]:
from statistics import mean

north_name, north_readings, north_unit = "Tromso", [-4.1, -2.6, -3.8], "C"
south_name, south_readings, south_unit = "Malaga", [66.0, 67.1, 68.2], "F"

print(north_name, north_readings, north_unit)
print(south_name, south_readings, south_unit)


Tromso [-4.1, -2.6, -3.8] C
Malaga [66.0, 67.1, 68.2] F


## Worked examples

### The program, written with functions

Three functions, each taking the values it needs. `in_celsius` is where the unit matters: a
Fahrenheit reading has to be converted before it can be compared with a Celsius one.

Read the calls at the bottom. Each one repeats the same three variables in the same order, and
that repetition is the thing to watch.


In [2]:
def average(readings, unit):
    return f"{mean(readings):.1f} {unit}"


def in_celsius(readings, unit):
    if unit == "C":
        return list(readings)
    return [round((r - 32) * 5 / 9, 1) for r in readings]


def report(name, readings, unit):
    return f"{name}: {average(readings, unit)}, coldest {min(in_celsius(readings, unit))} C"


print(report(north_name, north_readings, north_unit))
print(report(south_name, south_readings, south_unit))


Tromso: -3.5 C, coldest -4.1 C
Malaga: 67.1 F, coldest 18.9 C


Both lines are correct. Tromso averaged -3.5 C, and Malaga averaged 67.1 F, which is 18.9 C at its
coldest.

### The mistake the first version allows

Now call `report` with one station's readings and the other station's unit. This is a plausible
typo: the variables are adjacent in the source, they are the same shape, and the names differ by
one word.


In [3]:
print(report(north_name, north_readings, south_unit))
print(report(south_name, south_readings, north_unit))


Tromso: -3.5 F, coldest -20.1 C
Malaga: 67.1 C, coldest 66.0 C


No error, no warning, two wrong answers.

The first line reports a Norwegian station at -20.1 C when the coldest reading was -4.1 C, because
`in_celsius` was told the readings were Fahrenheit and converted numbers that needed no
converting. The second reports Malaga at 66.0 C, which is not a temperature that occurs in Spain.

Both numbers are the right shape, printed in the right format, in a line that looks exactly like a
correct one. Nothing downstream would catch it.

The three variables describe one station, and the language has no way to know that. Keeping them
in step is a rule that exists only in your head.

### The same program as a class

`__init__` runs when the object is created and stores the three values on it. Each method then
reaches for what it needs through `self` instead of taking it as an argument.

The mechanics belong to the **Your First Class** and **Methods** notebooks. For now, read this for its
shape: the three values go in once, at the top, and no method takes them again.


In [4]:
class Station:
    def __init__(self, name, readings, unit):
        self.name = name
        self.readings = readings
        self.unit = unit

    def average(self):
        return f"{mean(self.readings):.1f} {self.unit}"

    def in_celsius(self):
        if self.unit == "C":
            return list(self.readings)
        return [round((r - 32) * 5 / 9, 1) for r in self.readings]

    def report(self):
        return f"{self.name}: {self.average()}, coldest {min(self.in_celsius())} C"


north = Station("Tromso", [-4.1, -2.6, -3.8], "C")
south = Station("Malaga", [66.0, 67.1, 68.2], "F")

print(north.report())
print(south.report())


Tromso: -3.5 C, coldest -4.1 C
Malaga: 67.1 F, coldest 18.9 C


The same two lines as before.

### What the class actually bought

Try to make the earlier mistake. `north.report()` takes no arguments, so there is nowhere to put
the wrong unit. The only unit `north.in_celsius` can reach is `self.unit`, and for `north` that is
`"C"`.


In [5]:
print("north.unit:", north.unit)
print("south.unit:", south.unit)
print("north.report():", north.report())


north.unit: C
south.unit: F
north.report(): Tromso: -3.5 C, coldest -4.1 C


The mismatch is not detected. It is not expressible. That is a stronger guarantee than any check
you could add to the function version, because a check can be forgotten and this cannot.

This is the whole argument for classes, and it is worth stating plainly: a class is useful when it
removes a way of being wrong.

### An object is a value

The second gain is that a station is now one thing, so it can go wherever a value goes: into a
list, through `sorted`, into `min`, in and out of functions.


In [6]:
stations = [north, south, Station("Galway", [11.5, 12.1], "C")]

for station in sorted(stations, key=lambda s: min(s.in_celsius())):
    print(" ", station.report())

print("coldest:", min(stations, key=lambda s: min(s.in_celsius())).name)


  Tromso: -3.5 C, coldest -4.1 C
  Galway: 11.8 C, coldest 11.5 C
  Malaga: 67.1 F, coldest 18.9 C
coldest: Tromso


Doing this with loose variables means keeping three parallel lists, `names`, `readings_list` and
`units`, and trusting that index 2 in one lines up with index 2 in the others. Sorting one of them
would break that alignment silently. This is the same failure as before, one level up.


### When a function is better: one method and no state

Here is the first shape that should not be a class. The object is created, one method is called on
it, and it is discarded. Nothing is remembered between calls.


In [7]:
class ReadingValidator:
    def __init__(self, low=-90.0, high=60.0):
        self.low = low
        self.high = high

    def validate(self, reading):
        return self.low <= reading <= self.high


checker = ReadingValidator()
print("class:   ", [checker.validate(r) for r in [-4.1, 999.0]])


def valid_reading(reading, low=-90.0, high=60.0):
    return low <= reading <= high


print("function:", [valid_reading(r) for r in [-4.1, 999.0]])


class:    [True, False]
function: [True, False]


Eight lines and two lines, doing the same work.

The class version also has to be constructed before it can be used, so every caller writes an
extra line that changes nothing about the result. The test is whether the object holds anything between calls that the
caller would otherwise have to pass. Here it holds two numbers that have defaults, which is what a
default argument is for.

The exception is a validator that is genuinely configured once and used many times in many places,
where carrying the configuration is the point. Two numbers with sensible defaults is not that.

### When a dictionary is better: only data, and no behavior

The second shape is a class whose methods do nothing but hand back what was put in.


In [8]:
class StationRecord:
    def __init__(self, name, country):
        self._name = name
        self._country = country

    def get_name(self):
        return self._name

    def set_name(self, value):
        self._name = value

    def get_country(self):
        return self._country


record = StationRecord("Tromso", "NO")
print("class:", record.get_name(), record.get_country())

record = {"name": "Tromso", "country": "NO"}
print("dict: ", record["name"], record["country"])


class: Tromso NO
dict:  Tromso NO


Twelve lines replaced by one. `get_name` and `set_name` add no checking, no computation and no
constraint; they are `record.name` written the long way.

This pattern arrives from languages where a plain attribute cannot later be given behavior without
changing every caller. In Python it can: `@property` turns an attribute into a computed one while
callers keep writing `record.name`, which is the **Properties** notebook. So the getter is written
when it is needed, and not before.

If the thing really is only data but you still want a named type, comparison, and a readable
printout, the answer is neither of these. It is the **Dataclasses** notebook.

### When a module is better: a class used as a namespace

The third shape is a class that is never instantiated, holding constants that belong together.


In [9]:
class Settings:
    ARCHIVE = "readings.zip"
    UNITS = "C"
    LIMIT = 60.0


print("class:", Settings.ARCHIVE, Settings.UNITS, Settings.LIMIT)

SETTINGS = {"archive": "readings.zip", "units": "C", "limit": 60.0}
print("dict: ", SETTINGS["archive"], SETTINGS["units"], SETTINGS["limit"])


class: readings.zip C 60.0
dict:  readings.zip C 60.0


Both work, and the class version is not a disaster. It groups the names and reads acceptably.

What it does not do is anything a class is for. Nothing is ever constructed, there is one of it
forever, and no method operates on the data. A module already groups names, which the **Modules
and Imports** notebook covered: put those three lines in `settings.py` and `import settings` gives
you `settings.ARCHIVE` with no class at all.

Reach for the class here only when you want the grouping to be importable as one name from a
module that also contains other things.

### How to tell

No rule decides this for you, but the signals are reliable.

| What you notice | What it suggests |
|---|---|
| Several values always passed together, and always in step | A class |
| You need many of the thing, each with its own data | A class |
| Callers can currently pair the wrong values together | A class |
| One function, called on values that share no state | A function |
| The type would have attributes and no real methods | A dictionary, or a dataclass |
| The type would have methods and no attributes | A module of functions |
| You need exactly one of it, for the life of the program | A module |

The strongest signal is the third. If you can point at a wrong call that the language currently
permits, and a class would make that call impossible to write, the class is worth writing. If
you cannot, you are probably reaching for structure that the program does not need yet.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/01-why-classes-solutions.ipynb).

**1.** These three variables describe one book: `title = "Dune"`, `pages = 412`,
`read_so_far = 103`. Write a `Book` class holding all three, with a `progress()` method returning
the percentage read as a string like `25%`. Create two books and print the progress of each.


In [10]:
# your code here


**2.** Rewrite `StationRecord` so that it has no `get_` or `set_` methods, using plain attributes
instead. Create one, print both values, then change the name and print it again.


In [11]:
# your code here


**3.** Somebody hands you `class MathHelpers:` with a single method `def double(self, x): return
x * 2`. Write the version that should replace it, and use it on `[1, 2, 3]`.


In [12]:
# your code here


**4.** Add a fourth attribute to `Station`, `calibrated`, defaulting to `True`. Make `report()`
append ` (uncalibrated)` when it is `False`. Print a report for a calibrated station and an
uncalibrated one.


In [13]:
# your code here


**5.** Write a function `warmest(stations)` that takes a list of `Station` objects and returns the
name of the one with the highest mean reading. Run it on a list of three stations that all record
in Celsius.


In [14]:
# your code here


**6.** No code for this one. For each of the three situations below, write a comment saying class
or function, and one sentence of why.

- A program that draws one chart from one list of numbers and exits.
- A program tracking forty orders, each with an id, a customer, a list of items and a status.
- A program with six unrelated string-cleaning helpers used across four files.


In [15]:
# your code here


## Common errors

### TypeError: an argument `__init__` needed was not given

`Station("Bodo", [-2.6, -1.9])` looks like creating a station. It is calling `__init__`, which
takes three values after `self`, with only two.


In [16]:
Station("Bodo", [-2.6, -1.9])


TypeError: Station.__init__() missing 1 required positional argument: 'unit'

The message names the missing one, `unit`. Read the error as being about `__init__` rather than
about the class: the arguments you pass to `Station(...)` are the arguments `__init__` declares
after `self`.


In [17]:
print(Station("Bodo", [-2.6, -1.9], "C").report())


Bodo: -2.2 C, coldest -2.6 C


### AttributeError: a method reaches for something `__init__` never stored

A method can use any attribute name at all. Whether that name exists is not decided until the
method runs.


In [18]:
class Halfway:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def report(self):
        return f"{self.name}: {self.unit}"


Halfway("Bodo", [-2.6]).report()


AttributeError: 'Halfway' object has no attribute 'unit'

`self.unit` was never assigned, so there is nothing to read. Note that defining the class raised
nothing, and creating the object raised nothing. The error waits until `report` is actually called,
which is why a method never exercised by a test can carry a typo like this for a long time.


In [19]:
missing = Halfway("Bodo", [-2.6])

print("attributes it does have:", list(vars(missing)))


attributes it does have: ['name', 'readings']


`vars(object)` returns the attributes an object is carrying, which is the quickest way to see what
`__init__` actually stored.

### TypeError: the parentheses were left off

`Station` is the class. `Station(...)` is an object made from it. Assigning the first and then
calling a method on it produces a message that is confusing the first time.


In [20]:
maybe = Station

maybe.report()


TypeError: Station.report() missing 1 required positional argument: 'self'

`report() missing 1 required positional argument: 'self'` means a method was called without an
object to be called on. `north.report()` passes `north` as `self` automatically. `Station.report()`
has no object at all, so `self` is simply absent.

Seeing `self` named in an error message is almost always this: parentheses left off, or a method
called on the class instead of on an object.


In [21]:
print("the class: ", Station)
print("an object: ", Station("Bodo", [-2.6, -1.9], "C"))


the class:  <class '__main__.Station'>
an object:  <__main__.Station object at 0x10b509fd0>


The second line is not readable, and that is not a fault of yours. A default object prints as its
class and its memory address. The **Dunder Methods** notebook fixes it with `__repr__`.

### The quiet one: two names for one object

Assignment does not copy. `backup = original` makes a second name for the same object, exactly as
the **Lists** notebook showed for lists, and changing the object through one name changes what the
other name sees.


In [22]:
original = Station("Bodo", [-2.6, -1.9], "C")
backup = original

backup.readings.append(99.9)

print("backup.readings:  ", backup.readings)
print("original.readings:", original.readings)
print("same object:      ", original is backup)


backup.readings:   [-2.6, -1.9, 99.9]
original.readings: [-2.6, -1.9, 99.9]
same object:       True


Nothing raised, and `original` now has a reading nobody recorded.

The rule from **Lists** applies unchanged, because it was never really about lists: assignment
binds a name to a value, and objects are values like any other. `is` answers whether two names
label the same object, which is the check that shows what happened here.


## Recap

- A class binds data to the functions that operate on it, so the parts of one thing travel
  together.
- The strongest reason to write one is that it removes a way of being wrong, not that it shortens
  the code.
- Values that must stay in step, and that a caller can currently mismatch, are the signal.
- An object is a value, so it can go in a list, through `sorted`, and in and out of functions.
- `__init__` stores the data; methods reach it through `self` rather than taking it as arguments.
- A type with one method and no state is a function.
- A type with attributes and no behavior is a dictionary, or later a dataclass.
- A type that is never instantiated is a module.
- `vars(object)` shows what an object is actually carrying.
- `self` named in an error message usually means a method was called on the class, not an object.
- Assignment gives a second name, not a copy. Two names can share one object.


## What is next

The **Your First Class** notebook. This one showed a class without explaining it; that one builds
one line by line: what `class` does, what `__init__` is for, what `self` actually refers to, and
why every method takes it as its first argument.


---

[Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Your First Class](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/02-your-first-class.ipynb) &#8594;
